##CAJUPI SALES



Step 1 — Upload the files to Google Colab_

In [ ]:
import pandas as pd
import os

DATA_DIR = "data" if os.path.exists("data") else "."

def load_file(filename, **kwargs):
    path = os.path.join(DATA_DIR, filename)
    if not os.path.exists(path):
        # Running in Colab without the data/ folder: prompt for upload
        from google.colab import files
        uploaded = files.upload()
        path = filename
    return path


In [ ]:
invoices = pd.read_excel(load_file("fatura_qershor_dataset_v2.xlsx"))
items = pd.read_excel(load_file("shitje_produkte_lidhur_me_fatura.xlsx"))


In [ ]:
print(invoices.head())
print(items.head())

Step 2 — Basic cleaning and preparation

In [ ]:
import pandas as pd
import numpy as np

# convert date
invoices["date"] = pd.to_datetime(invoices["date"])
items["date"] = pd.to_datetime(items["date"])

# create hour column
invoices["hour"] = invoices["time"].str.slice(0, 2).astype(int)
items["hour"] = items["time"].str.slice(0, 2).astype(int)

# day name
invoices["day_name"] = invoices["date"].dt.day_name()
items["day_name"] = items["date"].dt.day_name()

# weekday/weekend flag
invoices["day_type"] = np.where(invoices["date"].dt.weekday < 5, "Weekday", "Weekend")
items["day_type"] = np.where(items["date"].dt.weekday < 5, "Weekday", "Weekend")

Step 3 — Analyze the invoice table

In [ ]:
total_revenue = invoices["price_lek"].sum()
total_revenue


In [ ]:
total_invoices = invoices["invoice_id"].nunique()
total_invoices

In [ ]:
avg_invoice = invoices["price_lek"].mean()
avg_invoice

In [ ]:
invoices["price_lek"].min(), invoices["price_lek"].max()

In [ ]:
daily_revenue = invoices.groupby("date")["price_lek"].sum().reset_index()
daily_revenue.head()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))
plt.plot(daily_revenue["date"], daily_revenue["price_lek"], marker="o")
plt.xticks(rotation=45)
plt.title("Të ardhurat ditore gjatë muajit qershor")
plt.xlabel("Data")
plt.ylabel("Të ardhurat (Lek)")
plt.tight_layout()
plt.show()

In [ ]:
hourly_revenue = invoices.groupby("hour")["price_lek"].sum().reset_index()
hourly_revenue

In [ ]:
plt.figure(figsize=(10,5))
plt.bar(hourly_revenue["hour"], hourly_revenue["price_lek"])
plt.title("Të ardhurat sipas orës")
plt.xlabel("Ora")
plt.ylabel("Të ardhurat (Lek)")
plt.xticks(hourly_revenue["hour"])
plt.tight_layout()
plt.show()

Step 6 — Number of invoices by hour

In [ ]:
hourly_invoices = invoices.groupby("hour")["invoice_id"].count().reset_index()
hourly_invoices

In [ ]:
plt.figure(figsize=(10,5))
plt.bar(hourly_invoices["hour"], hourly_invoices["invoice_id"])
plt.title("Numri i faturave sipas orës")
plt.xlabel("Ora")
plt.ylabel("Numri i faturave")
plt.xticks(hourly_invoices["hour"])
plt.tight_layout()
plt.show()

Step 7 — Weekday vs weekend

In [ ]:
daytype_revenue = invoices.groupby("day_type")["price_lek"].sum().reset_index()
daytype_revenue

In [ ]:
plt.figure(figsize=(6,4))
plt.bar(daytype_revenue["day_type"], daytype_revenue["price_lek"])
plt.title("Të ardhurat: Ditë jave vs Fundjavë")
plt.xlabel("Lloji i ditës")
plt.ylabel("Të ardhurat (Lek)")
plt.tight_layout()
plt.show()

In [ ]:
daytype_count = invoices.groupby("day_type")["invoice_id"].count().reset_index()
daytype_count

Step 8 — Product analysis

In [ ]:
product_sales = items.groupby("product_name")["quantity"].sum().sort_values(ascending=False).reset_index()
product_sales.head(10)

In [ ]:
top10 = product_sales.head(10)

plt.figure(figsize=(10,6))
plt.barh(top10["product_name"], top10["quantity"])
plt.gca().invert_yaxis()
plt.title("10 produktet më të shitura")
plt.xlabel("Sasia e shitur")
plt.ylabel("Produkti")
plt.tight_layout()
plt.show()

Step 9 — Product sales by hour

In [ ]:
product_hour = items.groupby(["hour", "product_name"])["quantity"].sum().reset_index()
product_hour.head()

In [ ]:
main_products = ["Pace", "Pilaf", "Kos", "Kafe", "Tasqebab", "Uje", "Raki", "Supe"]
filtered = product_hour[product_hour["product_name"].isin(main_products)]

In [ ]:
for product in main_products:
    temp = filtered[filtered["product_name"] == product]
    plt.figure(figsize=(8,4))
    plt.bar(temp["hour"], temp["quantity"])
    plt.title(f"Shitjet sipas orës për produktin: {product}")
    plt.xlabel("Ora")
    plt.ylabel("Sasia")
    plt.xticks(sorted(temp["hour"].unique()))
    plt.tight_layout()
    plt.show()

Step 10 — Morning vs afternoon product behavior

In [ ]:
items["time_period"] = np.where(items["hour"] < 12, "Morning", "Afternoon")

In [ ]:
period_product = items.groupby(["time_period", "product_name"])["quantity"].sum().reset_index()
period_product.head(20)

Step 11 — Merge both tables


In [ ]:
merged = items.merge(
    invoices[["invoice_id", "price_lek", "day_type", "hour"]],
    on="invoice_id",
    how="left"
)

merged.head()

BOTH TOGETHER

12.1 Products in higher-value invoices

In [ ]:
product_avg_bill = (
    merged.groupby("product_name")
    .agg(
        total_qty=("quantity", "sum"),
        avg_invoice_value=("price_lek", "mean"),
        invoice_count=("invoice_id", "nunique")
    )
    .sort_values("avg_invoice_value", ascending=False)
    .reset_index()
)

product_avg_bill.head(10)

12.2 Product sales on weekdays vs weekends

In [ ]:
product_daytype = (
    merged.groupby(["product_name", "day_type_x"])["quantity"]
    .sum()
    .reset_index()
)

product_daytype_pivot = product_daytype.pivot(
    index="product_name",
    columns="day_type_x",
    values="quantity"
).fillna(0)

product_daytype_pivot["Total"] = product_daytype_pivot.sum(axis=1)
product_daytype_pivot = product_daytype_pivot.sort_values("Total", ascending=False)

product_daytype_pivot.head(10)

In [ ]:
display(product_daytype_pivot.head(10))

#KORRIKU

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

june = pd.read_excel(load_file("fatura_qershor_dataset_v2.xlsx"))
july = pd.read_csv(load_file("fatura_korrik_dataset.csv"))


In [ ]:
june["date"] = pd.to_datetime(june["date"])
july["date"] = pd.to_datetime(july["date"])

june["hour"] = june["time"].str[:2].astype(int)
july["hour"] = july["time"].str[:2].astype(int)

june["day_type"] = np.where(june["date"].dt.weekday < 5, "Weekday", "Weekend")
july["day_type"] = np.where(july["date"].dt.weekday < 5, "Weekday", "Weekend")

Total revenue comparison

In [ ]:
comparison = pd.DataFrame({
    "Month": ["June", "July"],
    "Revenue": [june["price_lek"].sum(), july["price_lek"].sum()]
})

comparison

In [ ]:
plt.figure(figsize=(6,4))
plt.bar(comparison["Month"], comparison["Revenue"])
plt.title("Total Revenue: June vs July")
plt.ylabel("Revenue (Lek)")
plt.tight_layout()
plt.show()

Number of invoices comparison

In [ ]:
invoice_comp = pd.DataFrame({
    "Month": ["June", "July"],
    "Invoices": [june["invoice_id"].nunique(), july["invoice_id"].nunique()]
})

invoice_comp

In [ ]:
plt.figure(figsize=(6,4))
plt.bar(invoice_comp["Month"], invoice_comp["Invoices"])
plt.title("Number of Invoices: June vs July")
plt.ylabel("Number of Invoices")
plt.tight_layout()
plt.show()

Average invoice value comparison

In [ ]:
avg_comp = pd.DataFrame({
    "Month": ["June", "July"],
    "Average_Invoice": [june["price_lek"].mean(), july["price_lek"].mean()]
})

avg_comp

In [ ]:
plt.figure(figsize=(6,4))
plt.bar(avg_comp["Month"], avg_comp["Average_Invoice"])
plt.title("Average Invoice Value: June vs July")
plt.ylabel("Average Invoice (Lek)")
plt.tight_layout()
plt.show()

Revenue by hour comparison

In [ ]:
june_hour = june.groupby("hour")["price_lek"].sum().reset_index()
july_hour = july.groupby("hour")["price_lek"].sum().reset_index()

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(june_hour["hour"], june_hour["price_lek"], marker="o", label="June")
plt.plot(july_hour["hour"], july_hour["price_lek"], marker="o", label="July")
plt.title("Revenue by Hour: June vs July")
plt.xlabel("Hour")
plt.ylabel("Revenue (Lek)")
plt.legend()
plt.tight_layout()
plt.show()

Weekday vs weekend comparison

In [ ]:
june_daytype = june.groupby("day_type")["price_lek"].sum().reset_index()
july_daytype = july.groupby("day_type")["price_lek"].sum().reset_index()

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))

x = np.arange(2)
width = 0.35

ax.bar(x - width/2, june_daytype["price_lek"], width, label="June")
ax.bar(x + width/2, july_daytype["price_lek"], width, label="July")

ax.set_xticks(x)
ax.set_xticklabels(june_daytype["day_type"])
ax.set_title("Weekday vs Weekend Revenue")
ax.set_ylabel("Revenue (Lek)")
ax.legend()

plt.tight_layout()
plt.show()

for peak-hour revenue

In [ ]:
def is_peak(row):
    hour = int(row["time"][:2])
    minute = int(row["time"][3:5])
    return (
        (8 <= hour <= 9) or
        (hour == 12 and minute >= 30) or
        (hour == 13) or
        (hour == 14 and minute <= 30)
    )
june["peak"] = june.apply(is_peak, axis=1)
july["peak"] = july.apply(is_peak, axis=1)

peak_comp = pd.DataFrame({
    "Month": ["June", "July"],
    "Peak_Revenue": [
        june.loc[june["peak"], "price_lek"].sum(),
        july.loc[july["peak"], "price_lek"].sum()
    ]
})
peak_comp

In [ ]:
plt.figure(figsize=(6,4))
plt.bar(peak_comp["Month"], peak_comp["Peak_Revenue"])
plt.title("Peak-Hour Revenue: June vs July")
plt.ylabel("Revenue (Lek)")
plt.tight_layout()
plt.show()